In [1]:
import numpy as np
import heapq
import pandas as pd
from scipy.stats import skewnorm


In [2]:

# =====================================================
# PARAMETERS
# =====================================================

SIM_TIME             = 45 * 24
DRIVER_RATE          = 4.74
RIDER_RATE           = 34.6
PATIENCE_RATE        = 5
SPEED                = 20
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05

# Idea 3 parameter: weight on hub distance in matching score
LAMBDA = 0.5

# 5 hubs: Centre + N/S/E/W (from real BoxCar pickup density)
HUBS = [
    np.array([ 9.69, 10.40]),   # Centre
    np.array([ 8.98, 16.55]),   # North
    np.array([ 7.28,  4.09]),   # South
    np.array([16.05, 13.40]),   # East
    np.array([ 3.60, 11.69]),   # West
]

# Batch window for Idea 2 (minutes converted to hours)
BATCH_WINDOW = 1 / 60   # 1 minute


In [3]:

# =====================================================
# LOCATION FUNCTIONS
# =====================================================

def driver_initial_location():
    x = np.clip(np.random.normal(9.97,  4.36), 0, AREA_SIZE)
    y = np.clip(np.random.normal(11.51, 4.34), 0, AREA_SIZE)
    return np.array([x, y])

def rider_pickup_location():
    x = np.clip(skewnorm.rvs(a= 1.80, loc= 4.18, scale=5.97), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-2.49, loc=17.05, scale=6.31), 0, AREA_SIZE)
    return np.array([x, y])

def rider_dropoff_location():
    x = np.clip(skewnorm.rvs(a= -1.65, loc=15.51, scale=6.24), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-14.26, loc=19.50, scale=7.50), 0, AREA_SIZE)
    return np.array([x, y])

def nearest_hub_dist(location):
    return min(np.linalg.norm(location - h) for h in HUBS)


In [4]:

# =====================================================
# DRIVER & RIDER CLASSES
# =====================================================

class Driver:
    def __init__(self, driver_id, location, online_until):
        self.id           = driver_id
        self.location     = location
        self.online_until = online_until
        self.earnings     = 0
        self.status       = "idle"
        self.busy_time    = 0

class Rider:
    def __init__(self, rider_id, origin, destination, arrival_time, abandon_time):
        self.id           = rider_id
        self.origin       = origin
        self.destination  = destination
        self.arrival_time = arrival_time
        self.abandon_time = abandon_time
        self.matched      = False


In [ ]:

# =====================================================
# BASE SIMULATION (Baseline — closest distance match)
# =====================================================

class Simulation:

    def __init__(self):
        self.current_time     = 0
        self.event_list       = []
        self.idle_drivers     = {}
        self.busy_drivers     = {}
        self.waiting_riders   = {}
        self.driver_counter   = 0
        self.rider_counter    = 0
        self.total_riders     = 0
        self.abandoned_riders = 0
        self.total_wait_time  = 0
        self.driver_log       = []

    def exp_time(self, rate):
        return np.random.exponential(1 / rate)

    def dist(self, a, b):
        return np.linalg.norm(np.array(a) - np.array(b))

    def travel_time(self, d):
        mu = d / SPEED
        return np.random.uniform(0.8 * mu, 1.2 * mu)

    def schedule(self, t, e, d=None):
        heapq.heappush(self.event_list, (t, e, d))

    # --------------------------------------------------
    # MATCHING — override in subclasses
    # --------------------------------------------------

    def select_rider(self, driver):
        """Baseline: pick closest rider to driver."""
        return min(
            self.waiting_riders,
            key=lambda r: self.dist(driver.location,
                                    self.waiting_riders[r].origin)
        )

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders:
            return

        driver_id = next(iter(self.idle_drivers))
        driver    = self.idle_drivers[driver_id]
        rider_id  = self.select_rider(driver)

        rider         = self.waiting_riders[rider_id]
        rider.matched = True
        self.total_wait_time += self.current_time - rider.arrival_time

        pickup_dist = self.dist(driver.location, rider.origin)
        trip_dist   = self.dist(rider.origin, rider.destination)
        tt          = self.travel_time(pickup_dist) + self.travel_time(trip_dist)
        earnings    = (BASE_FARE + FARE_PER_MILE * trip_dist) - \
                      PETROL_COST_PER_MILE * (pickup_dist + trip_dist)

        driver.status     = "busy"
        driver.busy_time += tt
        self.busy_drivers[driver_id] = driver
        del self.idle_drivers[driver_id]
        del self.waiting_riders[rider_id]

        self.schedule(self.current_time + tt, "DROPOFF",
                      (driver_id, rider.destination, earnings))

    # --------------------------------------------------
    # EVENT HANDLERS
    # --------------------------------------------------

    def handle_driver_arrival(self):
        self.driver_counter += 1
        did          = self.driver_counter
        loc          = driver_initial_location()
        online_until = self.current_time + np.random.uniform(6, 8)

        driver = Driver(did, loc, online_until)
        self.idle_drivers[did] = driver
        self.driver_log.append({
            "driver_id": did, "arrival_time": self.current_time,
            "online_until": online_until, "earnings": 0, "busy_time": 0
        })
        self.schedule(online_until, "DRIVER_OFFLINE", did)
        self.schedule(self.current_time + self.exp_time(DRIVER_RATE), "DRIVER_ARRIVAL")
        self.try_match()

    def handle_rider_arrival(self):
        self.rider_counter += 1
        rid = self.rider_counter
        self.total_riders += 1
        origin       = rider_pickup_location()
        dest         = rider_dropoff_location()
        abandon_time = self.current_time + self.exp_time(PATIENCE_RATE)
        self.waiting_riders[rid] = Rider(rid, origin, dest,
                                         self.current_time, abandon_time)
        self.schedule(abandon_time, "RIDER_ABANDON", rid)
        self.schedule(self.current_time + self.exp_time(RIDER_RATE), "RIDER_ARRIVAL")
        self.try_match()

    def handle_dropoff(self, data):
        did, new_loc, earnings = data
        driver = self.busy_drivers[did]
        driver.earnings += earnings
        driver.location  = new_loc
        driver.status    = "idle"
        for d in self.driver_log:
            if d["driver_id"] == did:
                d["earnings"] += earnings; d["busy_time"] = driver.busy_time; break
        del self.busy_drivers[did]
        if self.current_time < driver.online_until:
            self.idle_drivers[did] = driver
        self.try_match()

    def run(self):
        self.schedule(0, "DRIVER_ARRIVAL")
        self.schedule(0, "RIDER_ARRIVAL")
        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, et, data = heapq.heappop(self.event_list)
            if   et == "DRIVER_ARRIVAL": self.handle_driver_arrival()
            elif et == "RIDER_ARRIVAL":  self.handle_rider_arrival()
            elif et == "DROPOFF":        self.handle_dropoff(data)
            elif et == "RIDER_ABANDON":
                if data in self.waiting_riders:
                    self.abandoned_riders += 1; del self.waiting_riders[data]
            elif et == "DRIVER_OFFLINE":
                self.idle_drivers.pop(data, None)
        return self.collect_results()

    def collect_results(self):
        matched = self.total_riders - self.abandoned_riders
        df      = pd.DataFrame(self.driver_log)
        df["online_duration"]  = df["online_until"] - df["arrival_time"]
        df["busy_time_capped"] = df[["busy_time","online_duration"]].min(axis=1)
        df["idle_time"]        = df["online_duration"] - df["busy_time_capped"]
        p  = df["earnings"].values; N = len(p); mp = p.mean()
        vp = np.sum((p - mp)**2) / (N - 1) if N > 1 else 0
        ps = np.sort(p)
        cvar   = np.mean(ps[:max(1, int(np.floor(CVAR_ALPHA * N)))])
        cvar_m = np.mean(ps[:max(1, int(np.floor(0.5 * N)))])
        return {
            "abandonment_rate" : self.abandoned_riders / self.total_riders,
            "avg_wait"         : self.total_wait_time / matched if matched > 0 else 0,
            "matched_riders"   : matched,
            "total_riders"     : self.total_riders,
            "abandoned_riders" : self.abandoned_riders,
            "total_drivers"    : N,
            "avg_earnings"     : mp,
            "avg_profit_per_hr": df["earnings"].sum() / df["online_duration"].sum(),
            "avg_idle_time"    : df["idle_time"].mean(),
            "fairness_cv"      : np.sqrt(vp) / mp if mp > 0 else 0,
            "cvar"             : cvar,
            "delta_cvar"       : cvar - cvar_m,
        }


# =====================================================
# IDEA 1 — Patience-Based Matching
#
# score = distance(driver → rider) / remaining_patience
# Pick rider with LOWEST score = most urgent + closest
# =====================================================

class SimulationIdea1(Simulation):

    def select_rider(self, driver):
        """
        Score = dist(driver → rider) / remaining patience time.
        Lowest score = rider who is closest AND running out of patience.
        """
        def score(rid):
            rider             = self.waiting_riders[rid]
            dist_to_rider     = self.dist(driver.location, rider.origin)
            remaining_patience = max(rider.abandon_time - self.current_time, 1e-6)
            return dist_to_rider / remaining_patience

        return min(self.waiting_riders, key=score)


# =====================================================
# IDEA 2 — Batch Matching (Total Distance Minimisation)
#
# Every BATCH_WINDOW minutes, collect all idle drivers
# and waiting riders, then solve a greedy assignment
# that minimises total pickup distance across all pairs.
# No instant matching — drivers and riders wait for
# the next batch window.
# =====================================================

class SimulationIdea2(Simulation):

    def __init__(self):
        super().__init__()
        self.batch_pending = False   # avoid scheduling duplicate batch events

    def try_match(self):
        """
        Override: do NOT match instantly.
        Schedule a batch event instead (if not already pending).
        """
        if not self.batch_pending and self.idle_drivers and self.waiting_riders:
            self.batch_pending = True
            self.schedule(self.current_time + BATCH_WINDOW, "BATCH_MATCH")

    def run_batch(self):
        """
        Greedy total-distance minimisation:
        Repeatedly pick the (driver, rider) pair with the
        shortest distance and assign them, until no more
        drivers or riders remain.
        """
        self.batch_pending = False

        drivers = list(self.idle_drivers.keys())
        riders  = list(self.waiting_riders.keys())

        if not drivers or not riders:
            return

        # Build distance matrix
        assigned_drivers = set()
        assigned_riders  = set()

        # Collect all pairs and sort by distance
        pairs = []
        for did in drivers:
            for rid in riders:
                d = self.dist(self.idle_drivers[did].location,
                              self.waiting_riders[rid].origin)
                pairs.append((d, did, rid))
        pairs.sort()

        for dist_val, did, rid in pairs:
            if did in assigned_drivers or rid in assigned_riders:
                continue
            if did not in self.idle_drivers or rid not in self.waiting_riders:
                continue

            assigned_drivers.add(did)
            assigned_riders.add(rid)

            driver        = self.idle_drivers[did]
            rider         = self.waiting_riders[rid]
            rider.matched = True
            self.total_wait_time += self.current_time - rider.arrival_time

            pickup_dist = self.dist(driver.location, rider.origin)
            trip_dist   = self.dist(rider.origin, rider.destination)
            tt          = self.travel_time(pickup_dist) + self.travel_time(trip_dist)
            earnings    = (BASE_FARE + FARE_PER_MILE * trip_dist) - \
                          PETROL_COST_PER_MILE * (pickup_dist + trip_dist)

            driver.status     = "busy"
            driver.busy_time += tt
            self.busy_drivers[did] = driver
            del self.idle_drivers[did]
            del self.waiting_riders[rid]

            self.schedule(self.current_time + tt, "DROPOFF",
                          (did, rider.destination, earnings))

        # If unmatched riders/drivers remain, schedule next batch
        if self.idle_drivers and self.waiting_riders:
            self.batch_pending = True
            self.schedule(self.current_time + BATCH_WINDOW, "BATCH_MATCH")

    def run(self):
        self.schedule(0, "DRIVER_ARRIVAL")
        self.schedule(0, "RIDER_ARRIVAL")
        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, et, data = heapq.heappop(self.event_list)
            if   et == "DRIVER_ARRIVAL": self.handle_driver_arrival()
            elif et == "RIDER_ARRIVAL":  self.handle_rider_arrival()
            elif et == "DROPOFF":        self.handle_dropoff(data)
            elif et == "BATCH_MATCH":    self.run_batch()
            elif et == "RIDER_ABANDON":
                if data in self.waiting_riders:
                    self.abandoned_riders += 1; del self.waiting_riders[data]
            elif et == "DRIVER_OFFLINE":
                self.idle_drivers.pop(data, None)
        return self.collect_results()


# =====================================================
# IDEA 3 — Hub-Penalised Matching
#
# When matching, pick the rider that minimises:
#   score = dist(driver → rider) + λ × dist(rider → nearest hub)
#
# λ > 0 → prefer riders who are close to a hub
#          so driver ends up near demand after dropoff
# =====================================================

class SimulationIdea3(Simulation):

    def __init__(self, lam=LAMBDA):
        super().__init__()
        self.lam = lam

    def select_rider(self, driver):
        """
        Score = dist(driver → rider) + λ × dist(rider's pickup → nearest hub).
        Prefer riders who are close AND near a hub.
        """
        def score(rid):
            rider         = self.waiting_riders[rid]
            dist_to_rider = self.dist(driver.location, rider.origin)
            hub_dist      = nearest_hub_dist(rider.origin)
            return dist_to_rider + self.lam * hub_dist

        return min(self.waiting_riders, key=score)


# =====================================================
# RUN ALL 4 AND COMPARE
# =====================================================

configs = [
    ("Baseline",       Simulation,       {}),
    ("Idea 1: Patience", SimulationIdea1, {}),
    ("Idea 2: Batch",    SimulationIdea2, {}),
    ("Idea 3: λ=0.3",   SimulationIdea3, {"lam": 0.3}),
    ("Idea 3: λ=0.5",   SimulationIdea3, {"lam": 0.5}),
    ("Idea 3: λ=1.0",   SimulationIdea3, {"lam": 1.0}),
]

results = {}
for label, SimClass, kwargs in configs:
    np.random.seed(42)
    print(f"Running: {label}...")
    results[label] = SimClass(**kwargs).run()
    r = results[label]
    print(f"  Abandon: {r['abandonment_rate']*100:.2f}%  "
          f"Wait: {r['avg_wait']*60:.3f} min  "
          f"Earn: £{r['avg_earnings']:.2f}  "
          f"CV: {r['fairness_cv']:.4f}")

# ── Comparison table ─────────────────────────────────────
b = results["Baseline"]
print("\n" + "="*85)
print(f"  {'Config':<22} {'Abandon%':>9} {'Wait(min)':>10} {'AvgEarn£':>9} "
      f"{'£/hr':>7} {'CV':>7} {'CVaR£':>8}")
print("-"*85)
for label, _, _ in configs:
    r = results[label]
    diff = ""
    if label != "Baseline":
        d = r['abandonment_rate'] - b['abandonment_rate']
        diff = f"({'↓' if d<0 else '↑'}{abs(d)*100:.2f}pp)"
    print(f"  {label:<22} {r['abandonment_rate']*100:>8.2f}% "
          f"{r['avg_wait']*60:>10.3f} "
          f"{r['avg_earnings']:>9.2f} "
          f"{r['avg_profit_per_hr']:>7.2f} "
          f"{r['fairness_cv']:>7.4f} "
          f"{r['cvar']:>8.2f}  {diff}")
print("="*85)

In [6]:

# =====================================================
# IDEA 1 — Patience-Based Matching
#
# score = distance(driver → rider) / remaining_patience
# Pick rider with LOWEST score = most urgent + closest
# =====================================================

class SimulationIdea1(Simulation):

    def select_rider(self, driver):
        """
        Score = dist(driver → rider) / remaining patience time.
        Lowest score = rider who is closest AND running out of patience.
        """
        def score(rid):
            rider             = self.waiting_riders[rid]
            dist_to_rider     = self.dist(driver.location, rider.origin)
            remaining_patience = max(rider.abandon_time - self.current_time, 1e-6)
            return dist_to_rider / remaining_patience

        return min(self.waiting_riders, key=score)



In [ ]:

# =====================================================
# IDEA 2 — Batch Matching (Total Distance Minimisation)
#
# Every BATCH_WINDOW minutes, collect all idle drivers
# and waiting riders, then solve a greedy assignment
# that minimises total pickup distance across all pairs.
# No instant matching — drivers and riders wait for
# the next batch window.
# =====================================================

class SimulationIdea2(Simulation):

    def __init__(self):
        super().__init__()
        self.batch_pending = False   # avoid scheduling duplicate batch events

    def try_match(self):
        """
        Override: do NOT match instantly.
        Schedule a batch event instead (if not already pending).
        """
        if not self.batch_pending and self.idle_drivers and self.waiting_riders:
            self.batch_pending = True
            self.schedule(self.current_time + BATCH_WINDOW, "BATCH_MATCH")

    def run_batch(self):
        """
        Greedy total-distance minimisation:
        Repeatedly pick the (driver, rider) pair with the
        shortest distance and assign them, until no more
        drivers or riders remain.
        """
        self.batch_pending = False

        drivers = list(self.idle_drivers.keys())
        riders  = list(self.waiting_riders.keys())

        if not drivers or not riders:
            return

        # Build distance matrix
        assigned_drivers = set()
        assigned_riders  = set()

        # Collect all pairs and sort by distance
        pairs = []
        for did in drivers:
            for rid in riders:
                d = self.dist(self.idle_drivers[did].location,
                              self.waiting_riders[rid].origin)
                pairs.append((d, did, rid))
        pairs.sort()

        for dist_val, did, rid in pairs:
            if did in assigned_drivers or rid in assigned_riders:
                continue
            if did not in self.idle_drivers or rid not in self.waiting_riders:
                continue

            assigned_drivers.add(did)
            assigned_riders.add(rid)

            driver        = self.idle_drivers[did]
            rider         = self.waiting_riders[rid]
            rider.matched = True
            self.total_wait_time += self.current_time - rider.arrival_time

            pickup_dist = self.dist(driver.location, rider.origin)
            trip_dist   = self.dist(rider.origin, rider.destination)
            tt          = self.travel_time(pickup_dist) + self.travel_time(trip_dist)
            earnings    = (BASE_FARE + FARE_PER_MILE * trip_dist) - \
                          PETROL_COST_PER_MILE * (pickup_dist + trip_dist)

            driver.status     = "busy"
            driver.busy_time += tt
            self.busy_drivers[did] = driver
            del self.idle_drivers[did]
            del self.waiting_riders[rid]

            self.schedule(self.current_time + tt, "DROPOFF",
                          (did, rider.destination, earnings))

        # If unmatched riders/drivers remain, schedule next batch
        if self.idle_drivers and self.waiting_riders:
            self.batch_pending = True
            self.schedule(self.current_time + BATCH_WINDOW, "BATCH_MATCH")

    def run(self):
        self.schedule(0, "DRIVER_ARRIVAL")
        self.schedule(0, "RIDER_ARRIVAL")
        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, et, data = heapq.heappop(self.event_list)
            if   et == "DRIVER_ARRIVAL": self.handle_driver_arrival()
            elif et == "RIDER_ARRIVAL":  self.handle_rider_arrival()
            elif et == "DROPOFF":        self.handle_dropoff(data)
            elif et == "BATCH_MATCH":    self.run_batch()
            elif et == "RIDER_ABANDON":
                if data in self.waiting_riders:
                    self.abandoned_riders += 1; del self.waiting_riders[data]
            elif et == "DRIVER_OFFLINE":
                self.idle_drivers.pop(data, None)
        return self.collect_results()


# =====================================================
# IDEA 3 — Hub-Penalised Matching
#
# When matching, pick the rider that minimises:
#   score = dist(driver → rider) + λ × dist(rider → nearest hub)
#
# λ > 0 → prefer riders who are close to a hub
#          so driver ends up near demand after dropoff
# =====================================================

class SimulationIdea3(Simulation):

    def __init__(self, lam=LAMBDA):
        super().__init__()
        self.lam = lam

    def select_rider(self, driver):
        """
        Score = dist(driver → rider) + λ × dist(rider's pickup → nearest hub).
        Prefer riders who are close AND near a hub.
        """
        def score(rid):
            rider         = self.waiting_riders[rid]
            dist_to_rider = self.dist(driver.location, rider.origin)
            hub_dist      = nearest_hub_dist(rider.origin)
            return dist_to_rider + self.lam * hub_dist

        return min(self.waiting_riders, key=score)


# =====================================================
# RUN ALL 4 AND COMPARE
# =====================================================

configs = [
    ("Baseline",       Simulation,       {}),
    ("Idea 1: Patience", SimulationIdea1, {}),
    ("Idea 2: Batch",    SimulationIdea2, {}),
    ("Idea 3: λ=0.3",   SimulationIdea3, {"lam": 0.3}),
    ("Idea 3: λ=0.5",   SimulationIdea3, {"lam": 0.5}),
    ("Idea 3: λ=1.0",   SimulationIdea3, {"lam": 1.0}),
]

results = {}
for label, SimClass, kwargs in configs:
    np.random.seed(42)
    print(f"Running: {label}...")
    results[label] = SimClass(**kwargs).run()
    r = results[label]
    print(f"  Abandon: {r['abandonment_rate']*100:.2f}%  "
          f"Wait: {r['avg_wait']*60:.3f} min  "
          f"Earn: £{r['avg_earnings']:.2f}  "
          f"CV: {r['fairness_cv']:.4f}")

# ── Comparison table ─────────────────────────────────────
b = results["Baseline"]
print("\n" + "="*85)
print(f"  {'Config':<22} {'Abandon%':>9} {'Wait(min)':>10} {'AvgEarn£':>9} "
      f"{'£/hr':>7} {'CV':>7} {'CVaR£':>8}")
print("-"*85)
for label, _, _ in configs:
    r = results[label]
    diff = ""
    if label != "Baseline":
        d = r['abandonment_rate'] - b['abandonment_rate']
        diff = f"({'↓' if d<0 else '↑'}{abs(d)*100:.2f}pp)"
    print(f"  {label:<22} {r['abandonment_rate']*100:>8.2f}% "
          f"{r['avg_wait']*60:>10.3f} "
          f"{r['avg_earnings']:>9.2f} "
          f"{r['avg_profit_per_hr']:>7.2f} "
          f"{r['fairness_cv']:>7.4f} "
          f"{r['cvar']:>8.2f}  {diff}")
print("="*85)

In [8]:

# =====================================================
# IDEA 3 — Hub-Penalised Matching
#
# When matching, pick the rider that minimises:
#   score = dist(driver → rider) + λ × dist(rider → nearest hub)
#
# λ > 0 → prefer riders who are close to a hub
#          so driver ends up near demand after dropoff
# =====================================================

class SimulationIdea3(Simulation):

    def __init__(self, lam=LAMBDA):
        super().__init__()
        self.lam = lam

    def select_rider(self, driver):
        """
        Score = dist(driver → rider) + λ × dist(rider's pickup → nearest hub).
        Prefer riders who are close AND near a hub.
        """
        def score(rid):
            rider         = self.waiting_riders[rid]
            dist_to_rider = self.dist(driver.location, rider.origin)
            hub_dist      = nearest_hub_dist(rider.origin)
            return dist_to_rider + self.lam * hub_dist

        return min(self.waiting_riders, key=score)


# =====================================================
# RUN ALL 4 AND COMPARE
# =====================================================

configs = [
    ("Baseline",       Simulation,       {}),
    ("Idea 1: Patience", SimulationIdea1, {}),
    ("Idea 2: Batch",    SimulationIdea2, {}),
    ("Idea 3: λ=0.3",   SimulationIdea3, {"lam": 0.3}),
    ("Idea 3: λ=0.5",   SimulationIdea3, {"lam": 0.5}),
    ("Idea 3: λ=1.0",   SimulationIdea3, {"lam": 1.0}),
]

results = {}
for label, SimClass, kwargs in configs:
    np.random.seed(42)
    print(f"Running: {label}...")
    results[label] = SimClass(**kwargs).run()
    r = results[label]
    print(f"  Abandon: {r['abandonment_rate']*100:.2f}%  "
          f"Wait: {r['avg_wait']*60:.3f} min  "
          f"Earn: £{r['avg_earnings']:.2f}  "
          f"CV: {r['fairness_cv']:.4f}")

# ── Comparison table ─────────────────────────────────────
b = results["Baseline"]
print("\n" + "="*85)
print(f"  {'Config':<22} {'Abandon%':>9} {'Wait(min)':>10} {'AvgEarn£':>9} "
      f"{'£/hr':>7} {'CV':>7} {'CVaR£':>8}")
print("-"*85)
for label, _, _ in configs:
    r = results[label]
    diff = ""
    if label != "Baseline":
        d = r['abandonment_rate'] - b['abandonment_rate']
        diff = f"({'↓' if d<0 else '↑'}{abs(d)*100:.2f}pp)"
    print(f"  {label:<22} {r['abandonment_rate']*100:>8.2f}% "
          f"{r['avg_wait']*60:>10.3f} "
          f"{r['avg_earnings']:>9.2f} "
          f"{r['avg_profit_per_hr']:>7.2f} "
          f"{r['fairness_cv']:>7.4f} "
          f"{r['cvar']:>8.2f}  {diff}")
print("="*85)

In [9]:

# =====================================================
# RUN ALL 4 AND COMPARE
# =====================================================

configs = [
    ("Baseline",       Simulation,       {}),
    ("Idea 1: Patience", SimulationIdea1, {}),
    ("Idea 2: Batch",    SimulationIdea2, {}),
    ("Idea 3: λ=0.3",   SimulationIdea3, {"lam": 0.3}),
    ("Idea 3: λ=0.5",   SimulationIdea3, {"lam": 0.5}),
    ("Idea 3: λ=1.0",   SimulationIdea3, {"lam": 1.0}),
]

results = {}
for label, SimClass, kwargs in configs:
    np.random.seed(42)
    print(f"Running: {label}...")
    results[label] = SimClass(**kwargs).run()
    r = results[label]
    print(f"  Abandon: {r['abandonment_rate']*100:.2f}%  "
          f"Wait: {r['avg_wait']*60:.3f} min  "
          f"Earn: £{r['avg_earnings']:.2f}  "
          f"CV: {r['fairness_cv']:.4f}")

# ── Comparison table ─────────────────────────────────────
b = results["Baseline"]
print("\n" + "="*85)
print(f"  {'Config':<22} {'Abandon%':>9} {'Wait(min)':>10} {'AvgEarn£':>9} "
      f"{'£/hr':>7} {'CV':>7} {'CVaR£':>8}")
print("-"*85)
for label, _, _ in configs:
    r = results[label]
    diff = ""
    if label != "Baseline":
        d = r['abandonment_rate'] - b['abandonment_rate']
        diff = f"({'↓' if d<0 else '↑'}{abs(d)*100:.2f}pp)"
    print(f"  {label:<22} {r['abandonment_rate']*100:>8.2f}% "
          f"{r['avg_wait']*60:>10.3f} "
          f"{r['avg_earnings']:>9.2f} "
          f"{r['avg_profit_per_hr']:>7.2f} "
          f"{r['fairness_cv']:>7.4f} "
          f"{r['cvar']:>8.2f}  {diff}")
print("="*85)

Running: Baseline...
  Abandon: 3.44%  Wait: 0.306 min  Earn: £111.77  CV: 0.1995
Running: Idea 1: Patience...
  Abandon: 3.88%  Wait: 0.196 min  Earn: £111.52  CV: 0.1989
Running: Idea 2: Batch...
  Abandon: 7.63%  Wait: 0.897 min  Earn: £111.51  CV: 0.3310
Running: Idea 3: λ=0.3...
  Abandon: 3.97%  Wait: 0.362 min  Earn: £114.87  CV: 0.1942
Running: Idea 3: λ=0.5...
  Abandon: 3.94%  Wait: 0.334 min  Earn: £113.80  CV: 0.1966
Running: Idea 3: λ=1.0...
  Abandon: 3.37%  Wait: 0.306 min  Earn: £112.66  CV: 0.1988

  Config                  Abandon%  Wait(min)  AvgEarn£    £/hr      CV    CVaR£
-------------------------------------------------------------------------------------
  Baseline                   3.44%      0.306    111.77   15.96  0.1995    66.95  
  Idea 1: Patience           3.88%      0.196    111.52   15.95  0.1989    66.65  (↑0.44pp)
  Idea 2: Batch              7.63%      0.897    111.51   15.96  0.3310    27.16  (↑4.19pp)
  Idea 3: λ=0.3              3.97%      0.362